# First Evaluation Run

Notebook nay la diem vao chinh de chay inference va danh gia. Logic load model, load dataset, extractor, metric va report nam trong `src/eval_pipeline/`.

In [ ]:
# Uncomment on a fresh notebook environment.
# %pip install -q -r ../requirements.txt

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "eval_pipeline").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing src/eval_pipeline")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

REPO_ROOT

In [ ]:
from eval_pipeline import EvaluationRequest, GenerationSettings, build_default_registry, evaluate_experiment


registry = build_default_registry()

SELECTED_MODELS = [
    "qwen3_mathqa_merged",
    "qwen3_gsm8k_merged",
    "qwen3_acereason_merged",
]

SELECTED_DATASETS = [
    "acereason_test",
    "gsm8k_test",
    "mathqa_test",
]

BATCH_SIZE = 2
MAX_NEW_TOKENS = 512
SEED = 42
RUN_NAME = "qwen3_first_eval"
DEFAULT_DATASET_LIMIT = None
PER_DATASET_LIMITS = {
    # "mathqa_test": 128,
}

SYSTEM_PROMPT_OVERRIDES = {
    # "gsm8k_test": "You are a precise math assistant. Think carefully and end with the final answer.",
}

# Add local paths here when you move between Colab, Kaggle, Run.ai or your workstation.
LOCAL_MODEL_OVERRIDES = {
    # "qwen3_mathqa_merged": ["/workspace/models/Qwen3-1.7B-mathqa-merged"],
    # "qwen3_gsm8k_merged": ["/workspace/models/qwen3-1.7b-gsm8k-merged"],
    # "qwen3_acereason_merged": ["/workspace/models/Qwen3-1.7B-acereason-merged"],
}

LOCAL_DATASET_OVERRIDES = {
    # "acereason_test": ["/workspace/datasets/AceReason-1.1-SFT-Filtered"],
    # "gsm8k_test": ["/workspace/datasets/gsm8k"],
    # "mathqa_test": ["/workspace/datasets/mathqa/test.json"],
}


def prepend_local_paths(spec_map, overrides):
    for name, extra_paths in overrides.items():
        if name not in spec_map:
            raise KeyError(f"Unknown registry key: {name}")
        spec_map[name].local_paths = list(extra_paths) + list(spec_map[name].local_paths)


prepend_local_paths(registry.models, LOCAL_MODEL_OVERRIDES)
prepend_local_paths(registry.datasets, LOCAL_DATASET_OVERRIDES)

for model_name in SELECTED_MODELS:
    print(model_name, "->", registry.models[model_name].local_paths, registry.models[model_name].hf_repo_id)

for dataset_name in SELECTED_DATASETS:
    print(dataset_name, "->", registry.datasets[dataset_name].local_paths, registry.datasets[dataset_name].hf_dataset_id)

In [ ]:
request = EvaluationRequest(
    model_names=SELECTED_MODELS,
    dataset_names=SELECTED_DATASETS,
    generation=GenerationSettings(
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
        repetition_penalty=1.05,
    ),
    seed=SEED,
    run_name=RUN_NAME,
    output_root=str(REPO_ROOT / "outputs" / "runs"),
    default_dataset_limit=DEFAULT_DATASET_LIMIT,
    per_dataset_limit=PER_DATASET_LIMITS,
    system_prompt_overrides=SYSTEM_PROMPT_OVERRIDES,
)

report = evaluate_experiment(
    request=request,
    registry=registry,
    project_root=REPO_ROOT,
)

report

In [ ]:
from pathlib import Path
from pprint import pprint

run_dir = Path(report["run_dir"])
print(f"Run directory: {run_dir}")
print("\nPair summaries:")
pprint(report["pair_summaries"])
print("\nFiles:")
for path in sorted(run_dir.rglob("*")):
    print(path.relative_to(run_dir))